# LinguaForge — GGUF Q4_K_M export

Merges the LinguaForge LoRA into the base Gemma 4 E4B weights, converts the
resulting model to GGUF, quantises it to **Q4_K_M**, and benchmarks CPU
decode tokens / sec inside the Kaggle T4 instance.

Outputs (all under `/kaggle/working/`):
- `merged_fp16/` — merged FP16 transformers checkpoint
- `gguf/linguaforge-gemma4-e4b.Q4_K_M.gguf` — quantised GGUF
- `gguf/Modelfile` — ready-to-run Ollama Modelfile
- `gguf/cpu_bench.json` — CPU decode benchmark

In [ ]:
import os
if not os.environ.get('HF_TOKEN'):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        pass
assert os.environ.get('HF_TOKEN'), 'HF_TOKEN missing — set as env var or Kaggle Secret before running.'
os.environ['HUGGING_FACE_HUB_TOKEN'] = os.environ['HF_TOKEN']
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
print('HF token loaded:', os.environ['HF_TOKEN'][:8], '...', os.environ['HF_TOKEN'][-4:])

In [ ]:
!pip install -qU unsloth==2026.5.2 peft transformers accelerate sentencepiece hf_transfer
!pip install -q gguf

## 1. Merge LoRA into base Gemma 4 E4B (FP16)

Use Unsloth's `save_pretrained_merged` so the merged weights match the
tokenizer layout the LoRA was trained against.

In [ ]:
import glob, os
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

# Mounted Kaggle dataset = same adapter, no HF Hub round-trip required.
candidates = glob.glob('/kaggle/input/**/adapter_model.safetensors', recursive=True)
assert candidates, 'Adapter dataset not mounted — add dongwei666/linguaforge-gemma4-204lang-lora as a data source.'
ADAPTER_DIR = os.path.dirname(candidates[0])
print('ADAPTER_DIR =', ADAPTER_DIR)
# /kaggle/working is only ~20 GB; merged FP16 + GGUF f16 is ~32 GB.
# Route the big intermediates through /tmp (~73 GB) and keep only the
# final Q4_K_M + Modelfile + bench under /kaggle/working.
MERGED_DIR = '/tmp/merged_fp16'
GGUF_TMP_DIR = '/tmp/gguf'
os.makedirs(GGUF_TMP_DIR, exist_ok=True)
os.makedirs('/kaggle/working/gguf', exist_ok=True)

model, tok = FastLanguageModel.from_pretrained(
    model_name=ADAPTER_DIR,
    max_seq_length=2048,
    load_in_4bit=True,
    token=os.environ['HF_TOKEN'],
)
tok = get_chat_template(tok, chat_template='gemma')
print('model + adapter loaded')

In [ ]:
# Merge LoRA + write FP16 checkpoint suitable for llama.cpp conversion.
model.save_pretrained_merged(MERGED_DIR, tok, save_method='merged_16bit')
!ls -lah {MERGED_DIR} | head -n 20

## 2. Build llama.cpp

In [ ]:
!cd /kaggle/working && git clone --depth 1 https://github.com/ggml-org/llama.cpp.git
!apt-get install -qq -y cmake build-essential libcurl4-openssl-dev > /tmp/apt.log 2>&1 || true
# CPU build only — Kaggle's CUDA toolkit headers aren't laid out the way
# llama.cpp's CUDA backend expects; v5 spent 8 min hitting CMake errors. The
# bench cell below uses a deliberately short -n so it finishes on CPU.
!cd /kaggle/working/llama.cpp && cmake -B build -DLLAMA_CURL=OFF -DGGML_NATIVE=ON > /tmp/cmake.log 2>&1
!tail -n 10 /tmp/cmake.log
!cd /kaggle/working/llama.cpp && cmake --build build --config Release -j$(nproc) --target llama-quantize llama-cli > /tmp/build.log 2>&1
!ls /kaggle/working/llama.cpp/build/bin | head

## 3. Convert merged FP16 → GGUF (FP16)


In [ ]:
!cd /kaggle/working/llama.cpp && python convert_hf_to_gguf.py {MERGED_DIR} \
    --outfile {GGUF_TMP_DIR}/linguaforge-gemma4-e4b.f16.gguf --outtype f16 2>&1 | tail -n 30

## 4. Quantise to Q4_K_M

In [ ]:
!/kaggle/working/llama.cpp/build/bin/llama-quantize \
    {GGUF_TMP_DIR}/linguaforge-gemma4-e4b.f16.gguf \
    /kaggle/working/gguf/linguaforge-gemma4-e4b.Q4_K_M.gguf Q4_K_M
# Free up the f16 intermediate immediately so disk doesn't fill mid-pipeline.
import os
if os.path.exists(f'{GGUF_TMP_DIR}/linguaforge-gemma4-e4b.f16.gguf'):
    os.remove(f'{GGUF_TMP_DIR}/linguaforge-gemma4-e4b.f16.gguf')
!ls -lah /kaggle/working/gguf

## 5. Ollama Modelfile (written *before* the bench so it's preserved even if bench OOMs)

In [ ]:
MODELFILE = '''\
FROM ./linguaforge-gemma4-e4b.Q4_K_M.gguf

TEMPLATE \"\"\"<start_of_turn>user
{{ if .System }}{{ .System }}
{{ end }}{{ .Prompt }}<end_of_turn>
<start_of_turn>model
{{ .Response }}<end_of_turn>
\"\"\"

PARAMETER stop \"<start_of_turn>\"
PARAMETER stop \"<end_of_turn>\"
PARAMETER temperature 0.7
PARAMETER top_p 0.9

SYSTEM \"\"\"You are LinguaForge, a multilingual tutor for endangered and low-resource languages.
Stay accurate, concise, and faithful to the target script. Always include
an English gloss in parentheses next to any native phrase.
\"\"\"
'''
with open('/kaggle/working/gguf/Modelfile', 'w') as f:
    f.write(MODELFILE)
print(open('/kaggle/working/gguf/Modelfile').read())

## 6. CPU benchmark inside Kaggle (best-effort; wrapped so a crash never blocks the deliverables)

In [ ]:
# Free RAM held by the Unsloth-loaded base model before running llama-cli;
# otherwise the OS kills the kernel when llama.cpp tries to mmap+load the
# 5 GB Q4_K_M file alongside the still-resident 5 GB of base weights.
try:
    del model, tok
except Exception:
    pass
import gc, torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

import json, re, subprocess, time, os
GGUF_PATH = '/kaggle/working/gguf/linguaforge-gemma4-e4b.Q4_K_M.gguf'
PROMPT = (
    '<start_of_turn>user\nTranslate this English sentence into Cherokee '
    '(Iroquoian, North America):\n\n'
    'The river remembers every footstep on its bank.\n'
    '<end_of_turn>\n<start_of_turn>model\n'
)

metrics = {'gguf_size_bytes': os.path.getsize(GGUF_PATH)}

def run_bench(extra_args, label, timeout_s, n_tokens):
    cmd = [
        '/kaggle/working/llama.cpp/build/bin/llama-cli',
        '-m', GGUF_PATH,
        '-no-cnv',
        '-n', str(n_tokens),
        '-p', PROMPT,
        '--seed', '1',
    ] + extra_args
    t0 = time.time()
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout_s)
        elapsed = time.time() - t0
        m = re.search(
            r'eval time =\s*([0-9.]+) ms /\s*(\d+) (?:tokens|runs)\s*\(\s*'
            r'([0-9.]+) ms per token,\s*([0-9.]+) tokens per second',
            result.stderr,
        )
        tps = float(m.group(4)) if m else None
        n_eval = int(m.group(2)) if m else None
        print(f'[{label}] elapsed={elapsed:.1f}s, tokens/sec={tps}, n_eval={n_eval}')
        print('--- stdout tail ---'); print(result.stdout[-600:])
        return {'elapsed_s': elapsed, 'tokens_per_sec': tps, 'eval_tokens': n_eval}
    except subprocess.TimeoutExpired:
        elapsed = time.time() - t0
        print(f'[{label}] TIMED OUT after {elapsed:.0f}s')
        return {'elapsed_s': elapsed, 'tokens_per_sec': None, 'eval_tokens': None, 'timeout': True}

# Single short CPU run with 16 tokens. Wrapped so a bench failure can
# never block the artifact-publishing step below — the GGUF + Modelfile
# are the important deliverables, the tokens/sec is a nice-to-have.
try:
    metrics['cpu_4threads_16tokens'] = run_bench(['-t', '4'], 'CPU 4t / 16 tok', 600, 16)
except Exception as e:
    print(f'bench raised: {type(e).__name__}: {e}')
    metrics['cpu_4threads_16tokens'] = {'error': repr(e)}

with open('/kaggle/working/gguf/bench.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print(json.dumps(metrics, indent=2))

In [ ]:
# Final disk hygiene: keep only the Q4_K_M GGUF, Modelfile, and bench JSON
# under /kaggle/working so the published kernel output stays small.
import shutil, os
for p in [
    '/kaggle/working/llama.cpp',
    MERGED_DIR,
    GGUF_TMP_DIR,
]:
    if os.path.isdir(p):
        shutil.rmtree(p, ignore_errors=True)
    elif os.path.isfile(p):
        os.remove(p)
!ls -lah /kaggle/working /kaggle/working/gguf